# exp_16 - baselines + depth ablation (track_c, Session 1)

Plan: docs/track_3_rl_reuse/session_plan.md. Three configs, single seed 42:
- tier0: 1 GAT layer, 12 feats (6 ERA5 vars + 6 cyclic time). Anchor.
- tier0b: 1 GAT layer, 15 feats (+lat/lon/elev). Static-feature effect vs tier0.
- depth2: 2 GAT layers, 12 feats. Depth (1 vs 2 hops) effect vs tier0.

Fixes vs exp_13: scaler train_end = 70%-split date; eval from best-val ckpt + train-val gap.
Per run we save full metrics, raw preds, PRIVATE extreme diagnostics, and an oversmoothing/embedding-diversity check (reusable for Tier 2A). Single seed now; multi-seed is Session 5.

In [1]:
import os, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [2]:
sys.path.append('C:/Users/rishe/Dissertation')

In [3]:
from utils.data_utils.data_helper_utils import (
    load_pivots, get_lat_lon_aligned, build_edge_index_radius, scale_pivots, temporal_split)
from utils.data_utils.dataset_files.gnn_dataset import (
    build_feature_tensor, build_time_features, add_time_features, SpatioTemporalDataset)
from models.gat_gru import GAT_GRU_Model
from models.gat_gru_multilayer import GAT_GRU_MultiLayer
from utils.train_utils import train_model, evaluate
from utils.metric_utils.metrics import rmse, mae, bias, nrmse
from utils.metric_utils.extreme_skill import extreme_skill_table
from utils.metric_utils.embedding_diag import embedding_report, last_gat_embeddings

In [4]:
SEED = 42  # single seed now (session_plan); Session 5 extends winners to [42,123,7]
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CONFIG = dict(epochs=30, lr=1e-3, batch_size=32, hidden_dim=64, heads=4, seq_len=7, horizon=1)
ROOT = 'C:/Users/rishe/Dissertation'
PIVOT_PATH = ROOT + '/data/era5_pivot_data/'
print('device', device, '| config', CONFIG)

device cuda | config {'epochs': 30, 'lr': 0.001, 'batch_size': 32, 'hidden_dim': 64, 'heads': 4, 'seq_len': 7, 'horizon': 1}


In [5]:
pivots = load_pivots(PIVOT_PATH)
station_df = pd.read_csv(ROOT + '/data/wb_station_coords.csv').rename(columns={'latitude':'lat','longitude':'lon'})
elev_df = pd.read_csv(ROOT + '/data/wb_station_elevation.csv')
print('rain pivot', pivots['rain'].shape, '| stations', station_df.shape, '| elevation', elev_df.shape)

rain pivot (19723, 291) | stations (293, 3) | elevation (293, 2)


In [6]:
lat, lon = get_lat_lon_aligned(pivots['rain'], station_df)
edge_index = build_edge_index_radius(lat, lon, threshold_km=100).to(device)
N = pivots['rain'].shape[1]
print('edge_index', tuple(edge_index.shape), '| N', N, '| avg degree %.2f' % (edge_index.shape[1]/N))

edge_index (2, 18110) | N 291 | avg degree 62.23


In [7]:
# FIX (status.md #2): scaler train_end == 70% split date (matches temporal_split)
dates = pivots['rain'].index
T = len(dates); t_end = int(0.7*T)
train_end_date = dates[t_end-1]
print('T', T, '| train_end_date', train_end_date)
scaled_pivots, scalers = scale_pivots(pivots, train_end=train_end_date)
mu, sigma = scalers['rain']
print('rain scaler mu=%.4f sigma=%.4f' % (mu, sigma))

T 19723 | train_end_date 2007-10-19 00:00:00
rain scaler mu=5.3051 sigma=12.5862


In [8]:
# Feature tensors. Feature 0 = rain (target). 12-feat = 6 vars + 6 cyclic time.
X_dyn, feature_order = build_feature_tensor(scaled_pivots, use_latent=True)
time_feats = build_time_features(dates)
X12 = add_time_features(X_dyn, time_feats)
# 15-feat: append z-scored lat/lon/elev as static features AFTER rain (so rain stays feature 0)
elev_map = dict(zip(elev_df['station_id'].astype(str), elev_df['elevation_m'].astype(float)))
assert not [c for c in pivots['rain'].columns if str(c) not in elev_map], 'elevation missing'
elev = np.array([elev_map[str(c)] for c in pivots['rain'].columns], dtype=float)
def _z(a):
    a = np.asarray(a, float); s = a.std(); return (a - a.mean())/(s if s else 1.0)
statics = np.stack([_z(lat), _z(lon), _z(elev)], axis=-1)
X15 = np.concatenate([X12, np.broadcast_to(statics[None,:,:], (X12.shape[0], N, 3))], axis=-1)
assert np.allclose(X15[...,0], X12[...,0])
print('X12', X12.shape, '| X15', X15.shape, '(static idx 12=lat 13=lon 14=elev, z-scored)')

X12 (19723, 291, 12) | X15 (19723, 291, 15) (static idx 12=lat 13=lon 14=elev, z-scored)


In [9]:
def real_metrics(yt, yp):
    return {'RMSE': rmse(yt,yp), 'MAE': mae(yt,yp), 'Bias': bias(yt,yp), 'NRMSE': nrmse(yt,yp)}
SEASONS = {'monsoon':[6,7,8,9], 'non_monsoon':[1,2,3,4,5,10,11,12]}
RESULT_ROOT = ROOT + '/experiments/results/exp_16_gat_gru_baseline_seeds'
SAVE_ROOT   = ROOT + '/experiments/saved_models/exp_16_gat_gru_baseline_seeds'
LOG_ROOT    = ROOT + '/experiments/logs/exp_16_gat_gru_baseline_seeds'
# GAT_GRU_MultiLayer(num_layers=1) == GAT_GRU_Model, so tier0 is the 1-layer point; depth2 adds one GAT layer.
RUNS = {
    'tier0':  dict(X=X12, in_channels=12, layers=1,
                   make=lambda: GAT_GRU_Model(in_channels=12, hidden_dim=64, heads=4),
                   desc='1-layer baseline 12-feat'),
    'tier0b': dict(X=X15, in_channels=15, layers=1,
                   make=lambda: GAT_GRU_Model(in_channels=15, hidden_dim=64, heads=4),
                   desc='1-layer +static 15-feat'),
    'depth2': dict(X=X12, in_channels=12, layers=2,
                   make=lambda: GAT_GRU_MultiLayer(in_channels=12, hidden_dim=64, heads=4, num_layers=2, residual=False),
                   desc='2-layer GAT 12-feat (depth vs tier0)'),
    # depth3 (if time): GAT_GRU_MultiLayer(12,64,heads=4,num_layers=3,residual=True)
}
SEEDS = [SEED]

In [ ]:
summary_rows = []
for cfg_name, cfg in RUNS.items():
    Xtr, Xva, Xte = temporal_split(cfg['X'], dates)
    train_loader = DataLoader(SpatioTemporalDataset(Xtr), batch_size=CONFIG['batch_size'], shuffle=True)
    val_loader   = DataLoader(SpatioTemporalDataset(Xva), batch_size=CONFIG['batch_size'])
    test_loader  = DataLoader(SpatioTemporalDataset(Xte), batch_size=CONFIG['batch_size'])
    print('\n===', cfg_name, cfg['desc'], '| splits', Xtr.shape, Xva.shape, Xte.shape, '===')
    for seed in SEEDS:
        torch.manual_seed(seed); np.random.seed(seed)
        exp_name = 'exp_16_' + cfg_name + '_seed' + str(seed)
        save_dir = SAVE_ROOT + '/' + cfg_name + '/seed_' + str(seed)
        log_dir  = LOG_ROOT  + '/' + cfg_name + '/seed_' + str(seed)
        res_dir  = RESULT_ROOT + '/' + cfg_name + '/seed_' + str(seed)
        for d in (save_dir, log_dir, res_dir): os.makedirs(d, exist_ok=True)
        model = cfg['make']()
        model = train_model(train_loader, val_loader, model, edge_index, device,
                            epochs=CONFIG['epochs'], lr=CONFIG['lr'], criterion=nn.MSELoss(),
                            save_dir=save_dir, log_dir=log_dir, experiment_name=exp_name)
        model.load_state_dict(torch.load(save_dir + '/' + exp_name + '_best.pt', map_location=device))
        crit = nn.MSELoss()
        _, preds, targets = evaluate(model, test_loader, crit, edge_index, device)
        train_loss, _, _  = evaluate(model, train_loader, crit, edge_index, device)
        val_loss, _, _    = evaluate(model, val_loader, crit, edge_index, device)
        gap = float(val_loss - train_loss)
        preds_real = preds*sigma + mu; targets_real = targets*sigma + mu
        np.savez_compressed(res_dir + '/test_predictions_real.npz', obs=targets_real, pred=preds_real)
        ov = real_metrics(targets_real.reshape(-1), preds_real.reshape(-1))
        pd.DataFrame([ov]).to_csv(res_dir + '/overall_metrics.csv', index=False)
        dt = dates[-len(Xte):]; months = dt.month.values[CONFIG['seq_len']:]
        months_flat = np.repeat(months[:,None], preds.shape[1], axis=1).reshape(-1)
        seas = []
        for sname, sm in SEASONS.items():
            m = np.isin(months_flat, sm)
            r = real_metrics(targets_real.reshape(-1)[m], preds_real.reshape(-1)[m]); r['season']=sname; seas.append(r)
        pd.DataFrame(seas).to_csv(res_dir + '/seasonal_metrics_real.csv', index=False)
        ids = pivots['rain'].columns.tolist()
        st = [dict(real_metrics(targets_real[:,i], preds_real[:,i]), station_id=ids[i]) for i in range(preds.shape[1])]
        pd.DataFrame(st).to_csv(res_dir + '/station_metrics_real.csv', index=False)
        ext = extreme_skill_table(targets_real.reshape(-1), preds_real.reshape(-1))
        ext.to_csv(res_dir + '/extreme_skill_real.csv', index=False)
        e645 = ext[np.isclose(ext['threshold_mm'], 64.5)].iloc[0]
        xb, _ = next(iter(test_loader))
        emb = last_gat_embeddings(model, xb, edge_index, device)
        ei_np = edge_index.detach().cpu().numpy()
        reps = [embedding_report(emb[b].cpu().numpy(), ei_np) for b in range(emb.shape[0])]
        edrep = {k: float(np.mean([r[k] for r in reps])) for k in reps[0]}
        pd.DataFrame([edrep]).to_csv(res_dir + '/embedding_diag.csv', index=False)
        summary_rows.append(dict(config=cfg_name, layers=cfg['layers'], in_channels=cfg['in_channels'],
            seed=seed, RMSE=ov['RMSE'], MAE=ov['MAE'], Bias=ov['Bias'], NRMSE=ov['NRMSE'],
            train_loss=float(train_loss), val_loss=float(val_loss), train_val_gap=gap,
            CSI_64p5=float(e645['CSI']), POD_64p5=float(e645['POD']), n_events_64p5=int(e645['n_events']),
            dirichlet_norm=edrep['dirichlet_energy_norm'], MAD_cosine=edrep['MAD_cosine'], eff_rank=edrep['effective_rank']))
        print(cfg_name, 'RMSE %.3f gap %.4f | CSI@64.5' % (ov['RMSE'], gap), e645['CSI'],
              '| emb Dir %.3f MAD %.3f rank %.1f' % (edrep['dirichlet_energy_norm'], edrep['MAD_cosine'], edrep['effective_rank']))
summary = pd.DataFrame(summary_rows)
summary.to_csv(RESULT_ROOT + '/comparison_seed' + str(SEED) + '.csv', index=False)
summary


=== tier0 1-layer baseline 12-feat | splits (13806, 291, 12) (2958, 291, 12) (2959, 291, 12) ===
2026-05-30 05:28:19 | INFO | Starting GCN+GRU training
2026-05-30 05:28:19 | INFO | Device: cuda
2026-05-30 05:28:19 | INFO | Experiment Config:
2026-05-30 05:28:19 | INFO | LR: 0.001
2026-05-30 05:28:19 | INFO | Epochs: 30
2026-05-30 05:28:19 | INFO | Criterion: MSELoss()


Train:   0%|          | 0/432 [00:00<?, ?it/s]

## How to read

- RMSE/MAE/Bias/NRMSE = primary. tier0b-tier0 = static-feature effect; depth2-tier0 = depth effect.
- train_val_gap = overfitting (val-train MSE at best ckpt); watch if depth2 widens it.
- embedding_diag.csv = oversmoothing check on last-GAT node embeddings. dirichlet_norm/MAD near 0 and eff_rank near 1 => collapsed/oversmoothed. Compare depth2 vs tier0; a big drop => the 2nd layer oversmooths (prefer 1 layer, or add residual for depth>=3). Caveat: these can lag RMSE - read together. Reused in Tier 2A.
- extreme_skill_real.csv = PRIVATE limitation diagnostic (expect low CSI@64.5). Not a headline (session_plan §0).

Single-seed exploratory; nothing final until Session 5 multi-seed. depth3: uncomment in RUNS if time permits.

In [ ]:
for cfg_name in RUNS:
    base = RESULT_ROOT + '/' + cfg_name + '/seed_' + str(SEED)
    p = base + '/embedding_diag.csv'
    if os.path.exists(p):
        print('---', cfg_name, 'embedding diagnostic ---'); print(pd.read_csv(p).to_string(index=False))